# NYC Restaurant Inspections → Survival Analysis Data Prep

This notebook takes raw [NYC DOHMH restaurant inspection data](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j) and reshapes it into a **survival analysis** dataset: one row per restaurant, with a start time, an outcome (closed or not), and how long it survived.

Output: `restaurant_survival.csv`, used in `02_cox_regression_rsf.ipynb`.

In [1]:
import pandas as pd

## Load raw inspection data

Read the DOHMH inspection export. Each row is one violation found during one inspection, so the same restaurant/date shows up many times.

In [ ]:
df = pd.read_csv("dohmh.csv", low_memory=False)

## Clean the core columns

Parse dates and scores, and drop rows with placeholder/invalid values so they don't distort the grouping and modeling steps later on.

In [ ]:
df["INSPECTION DATE"] = pd.to_datetime(df["INSPECTION DATE"], errors="coerce")
df = df[df["INSPECTION DATE"] > "1900-01-01"]           # drop never-inspected placeholders
df["SCORE"] = pd.to_numeric(df["SCORE"], errors="coerce")

# keep only the 5 real boroughs 
df = df[df["BORO"].isin(["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"])]

## Flag closures and violations

Derive row-level flags: whether the inspection resulted in a closure, and whether each violation is a food-safety issue vs. an administrative one (codes 15-22 are administrative, e.g. permit/signage issues).

In [4]:
df["is_closure"]  = df["ACTION"].astype(str).str.contains("closed", case=False, na=False)
df["has_viol"]    = df["VIOLATION CODE"].notna()
df["is_critical"] = df["CRITICAL FLAG"].astype(str).str.strip().eq("Critical")

# split violations: admin/special codes 15-22 vs food-safety (health). codes are like "15A".
code_num = pd.to_numeric(df["VIOLATION CODE"].astype(str).str.extract(r"^(\d+)")[0], errors="coerce")
df["is_admin_viol"]  = df["has_viol"] & code_num.between(15, 22)
df["is_health_viol"] = df["has_viol"] & ~code_num.between(15, 22)

## Collapse to one row per inspection

The raw data has one row per violation, so a single inspection can span several rows. Group by restaurant and date to get one row per inspection, with violation counts summed up.

In [5]:
insp = df.groupby(["CAMIS", "INSPECTION DATE"], as_index=False).agg(
    BORO=("BORO", "first"),
    CUISINE=("CUISINE DESCRIPTION", "first"),
    INSP_TYPE=("INSPECTION TYPE", "first"),
    SCORE=("SCORE", "max"),
    health_viol=("is_health_viol", "sum"),
    admin_viol=("is_admin_viol", "sum"),
    n_critical=("is_critical", "sum"),
    is_closure=("is_closure", "any"),
).sort_values(["CAMIS", "INSPECTION DATE"])

## Mark which inspections are "operational"

Only cycle and pre-permit inspections reflect a restaurant actually up and running (as opposed to e.g. a compliance re-check tied to a complaint). These are the candidates for a restaurant's baseline.

In [6]:
operational = [
    "Cycle Inspection / Initial Inspection",
    "Cycle Inspection / Re-inspection",
    "Pre-permit (Operational) / Initial Inspection",
    "Pre-permit (Operational) / Re-inspection",
]
insp["is_operational"] = insp["INSP_TYPE"].isin(operational)

## Build each restaurant's baseline ("time zero")

Use a restaurant's first operational inspection as its starting point, capturing the score and violation counts at that moment. This is the baseline every restaurant's survival is measured from.

In [7]:
baseline = (
    insp[insp["is_operational"]]
    .drop_duplicates("CAMIS", keep="first")
    .set_index("CAMIS")
    .rename(columns={"INSPECTION DATE": "time_zero", "SCORE": "init_score",
                     "health_viol": "init_health_viol", "admin_viol": "init_admin_viol"})
)
baseline["init_has_critical"] = (baseline["n_critical"] > 0).astype(int)
baseline = baseline[["BORO", "CUISINE", "time_zero", "init_score",
                     "init_health_viol", "init_admin_viol", "init_has_critical"]]

## Compute the survival outcome

For each restaurant, find the first closure after time zero (the "event"). If it was never closed, its duration is censored at the latest record date in the data instead.

In [8]:
first_closure = insp[insp["is_closure"]].groupby("CAMIS")["INSPECTION DATE"].min().rename("first_closure")

rec = pd.to_datetime(df["RECORD DATE"], errors="coerce")
cutoff = rec.max() if rec.notna().any() else insp["INSPECTION DATE"].max()

out = baseline.join(first_closure)
out["event"] = out["first_closure"].notna().astype(int)
out["duration_days"] = (out["first_closure"].fillna(cutoff) - out["time_zero"]).dt.days

## Sanity check and export

Drop the handful of rows with a zero/negative duration (data entry issues), save the final one-row-per-restaurant dataset, and print a quick summary.

In [9]:
print("non-positive:", (out["duration_days"] <= 0).sum(),
      "| negative:", (out["duration_days"] < 0).sum())     # negative = date bug
out = out[out["duration_days"] > 0].reset_index()

out.to_csv("restaurant_survival.csv", index=False)
print(f"{len(out):,} restaurants | {out['event'].sum():,} events "
      f"({out['event'].mean():.1%}) | median {out['duration_days'].median():.0f}d")

non-positive: 439 | negative: 19
26,534 restaurants | 845 events (3.2%) | median 951d
